## Get a list of all Folder names and its respective image names in the current directory

In [11]:
import os
import pandas as pd
from IPython.display import display

def display_files_no_extensions(base_folder):
    """
    Display folder names and filenames without extensions
    """
    try:
        if not os.path.isdir(base_folder):
            raise FileNotFoundError(f"Folder not found: {base_folder}")

        # Get subfolders
        subfolders = [f.name for f in os.scandir(base_folder) if f.is_dir()]
        
        if not subfolders:
            print(f"No subfolders found in: {base_folder}")
            return
            
        file_data = []
        
        for folder_name in subfolders:
            subfolder_path = os.path.join(base_folder, folder_name)
            
            try:
                for f in os.scandir(subfolder_path):
                    if f.is_file():
                        # Remove file extension
                        filename = os.path.splitext(f.name)[0]
                        # Replace hyphens with underscores in folder name
                        clean_folder_name = folder_name.replace('-', '_')
                        file_data.append((clean_folder_name, filename))
                
            except PermissionError:
                # Replace hyphens with underscores in folder name even for permission errors
                clean_folder_name = folder_name.replace('-', '_')
                file_data.append((clean_folder_name, "PERMISSION DENIED"))
        
        if file_data:
            # Create and display DataFrame without index
            files_df = pd.DataFrame(file_data, columns=['Folder', 'Filename'])
            
            # Display without index and with clean formatting
            display(files_df.style.hide(axis='index'))
            
            print(f"\nFound {len(file_data)} files in {len(subfolders)} subfolders")
        else:
            print("\nNo files found in any subfolders")
            
    except Exception as e:
        print(f"Error: {str(e)}")

# Run with your path
base_path = r"C:\Users\Biplov.Parajuli\Downloads\ImageOrderHeruntergeladen\ID1696 Bammental - LBDB"
display_files_no_extensions(base_path)

Folder,Filename
365_BS_2099_12,163
365_BS_2099_12,164
7_3095,3160
7_3095,3161
7_3187,4053
7_3187,4054
7_3188,4074
7_3188,4075



Found 8 files in 5 subfolders


## Renaming downloaded Images based on Import-Bilder excel sheet

### Read Bilderbestellungen_Fluginfos excel sheet

In [12]:
import pandas as pd

# Read Excel file
file_path = r'L:\_Luftbilddatenbank\ExcelHerkunft\Bilderbestellungen_Fluginfos.xlsm'
df = pd.read_excel(file_path, sheet_name='Import-Bilder')

# Display columns and first 10 rows
print("Columns:", df.columns.tolist())
print(f"\nShape: {df.shape}")
print(f"\nFirst 10 rows:")
display(df.head(10))

Columns: ['Bestell-ID', 'Massstab', 'Sortie original', 'Bildnummer', 'Flugnummer', 'Datum', 'Sortie#Bild', 'Sortie Landesamt', 'im Besitz', 'Ordner', 'Quelle', 'Bestellnummer/Rechnungsnummer', 'Projekt-Nr.', 'Status\n(Georeferenziert oder importiert)', 'Unnamed: 14', 'Remarks', 'Unnamed: 16']

Shape: (35156, 17)

First 10 rows:


,Bestell-ID,Massstab,Sortie original,Bildnummer,Flugnummer,Datum,Sortie#Bild,Sortie Landesamt,im Besitz,Ordner,Quelle,Bestellnummer/Rechnungsnummer,Projekt-Nr.,Status\n(Georeferenziert oder importiert),Unnamed: 14,Remarks,Unnamed: 16
0,0,Maßstab prüfen,LBDB,NaN,NaN,NaN,LBDB#,INVALID ENTRY,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,0,Maßstab prüfen,NaN,NaN,NaN,NaN,#,INVALID ENTRY,NaN,NaN,NaN,NaN,NaN,NaN,Bereits andere Version Georeferenziert,NaN,NaN
2,0,Maßstab prüfen,NaN,NaN,NaN,NaN,#,INVALID ENTRY,NaN,NaN,NaN,NaN,NaN,NaN,Georeferenziert & Rahmen gezeichnet,NaN,"Formel um eine ""0"" hinter dem Unterstrich hinz..."
3,0,Maßstab prüfen,NaN,NaN,NaN,NaN,#,INVALID ENTRY,NaN,NaN,NaN,NaN,NaN,NaN,Importiert,NaN,"! Liste ist keine Tabelle, sondern ein Filter,..."
4,0,Maßstab prüfen,NaN,NaN,NaN,NaN,#,INVALID ENTRY,NaN,NaN,NaN,NaN,NaN,NaN,Warten auf Lieferung,NaN,NaN
5,1,10000,US31_4056,1061.0,0,1945-01-29 00:00:00,US31_4056#1061,#_#US31_4056#ID0001#29_01_1945,2.0,Baustert_Trasse westlich Bitburg_I 2018-09-19,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,1,10000,US31_4056,1062.0,0,1945-01-29 00:00:00,US31_4056#1062,#_#US31_4056#ID0001#29_01_1945,2.0,Baustert_Trasse westlich Bitburg_I 2018-09-19,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,1,10000,US31_4056,1063.0,0,1945-01-29 00:00:00,US31_4056#1063,#_#US31_4056#ID0001#29_01_1945,2.0,Baustert_Trasse westlich Bitburg_I 2018-09-19,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,1,10000,US31_4056,1064.0,0,1945-01-29 00:00:00,US31_4056#1064,#_#US31_4056#ID0001#29_01_1945,2.0,Baustert_Trasse westlich Bitburg_I 2018-09-19,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,1,10000,US31_4056,1065.0,0,1945-01-29 00:00:00,US31_4056#1065,#_#US31_4056#ID0001#29_01_1945,2.0,Baustert_Trasse westlich Bitburg_I 2018-09-19,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Renaming Images based on Excel 

In [13]:
import pandas as pd
import os
import re
from difflib import SequenceMatcher
from IPython.display import display
from collections import defaultdict

def similarity_score(str1, str2):
    """Calculate similarity score between two strings (0-1)"""
    if pd.isna(str1) or pd.isna(str2):
        return 0
    return SequenceMatcher(None, str(str1).lower(), str(str2).lower()).ratio()

def find_all_sortie_matches(folder_sortie, excel_sorties):
    """Find all sortie matches with similarity scores"""
    matches = []
    for excel_sortie in excel_sorties:
        score = similarity_score(folder_sortie, excel_sortie)
        matches.append((excel_sortie, score))
    return sorted(matches, key=lambda x: x[1], reverse=True)

def find_image_files(base_path):
    """Recursively find all image files in any subfolder"""
    image_files = []
    for root, dirs, files in os.walk(base_path):
        for file in files:
            if file.lower().endswith(('.tif', '.jpg', '.jpeg', '.png', '.tiff')):
                full_path = os.path.join(root, file)
                parent_folder = os.path.basename(root)
                image_files.append((full_path, file, parent_folder))
    return image_files

# Read Excel data
excel_path = r'L:\_Luftbilddatenbank\ExcelHerkunft\Bilderbestellungen_Fluginfos.xlsm'
df = pd.read_excel(excel_path, sheet_name='Import-Bilder')

# Extract ID from folder path
folder_path = r'C:\Users\Biplov.Parajuli\Downloads\ImageOrderHeruntergeladen\ID1696 Bammental - LBDB'
bestell_id = re.search(r'ID(\d+)', folder_path).group(1)

print(f"📁 Extracted Bestell-ID: {bestell_id}")
print(f"📊 Total records in Excel: {len(df)}")

# Filter Excel data
id_data = df[df['Bestell-ID'] == int(bestell_id)]
print(f"🔍 Found {len(id_data)} records for ID {bestell_id}")

excel_sorties = id_data['Sortie original'].dropna().unique()
print(f"📋 Unique sorties in Excel: {list(excel_sorties)}\n")

# Find images
print("🔍 Searching for image files in all subfolders...")
all_image_files = find_image_files(folder_path)
print(f"📁 Found {len(all_image_files)} image files across all subfolders")

# Group by folder
images_by_sortie = defaultdict(list)
for img_path, filename, parent_folder in all_image_files:
    images_by_sortie[parent_folder].append((img_path, filename))

print(f"📂 Found {len(images_by_sortie)} unique parent folders (sorties)\n")

match_results = []
total_processed = 0
success_count = 0
error_count = 0

# Process each folder
for folder_sortie, image_list in images_by_sortie.items():
    print(f"📂 Processing Folder Sortie: {folder_sortie}")
    print(f"   📷 Found {len(image_list)} images")

    all_matches = find_all_sortie_matches(folder_sortie, excel_sorties)

    matches_df = pd.DataFrame(all_matches, columns=['Excel Sortie', 'Similarity'])
    matches_df['Folder Sortie'] = folder_sortie
    matches_df = matches_df[['Folder Sortie', 'Excel Sortie', 'Similarity']]

    print("   📊 All possible matches:")
    display(matches_df.style.format({'Similarity': '{:.1%}'}))

    best_match, best_similarity = all_matches[0] if all_matches else (None, 0)

    if best_match and best_similarity > 0:
        print(f"   ✅ Using best match: {best_match} ({best_similarity:.1%})")

        sortie_data = id_data[id_data['Sortie original'] == best_match]
        print(f"   📋 Found {len(sortie_data)} Excel records")

        for img_path, filename in image_list:
            total_processed += 1

            img_num = os.path.splitext(filename)[0]
            file_ext = os.path.splitext(filename)[1]

            try:
                img_num_int = int(img_num)
                formatted_num = f"{img_num_int:04d}"  # ✅ 4-digit formatting
                match = sortie_data[sortie_data['Bildnummer'] == img_num_int]
            except ValueError:
                formatted_num = img_num
                match = pd.DataFrame()

            match_found = not match.empty
            sortie_landesamt = match.iloc[0]['Sortie Landesamt'] if match_found else "NO MATCH"

            match_results.append({
                'Folder Sortie': folder_sortie,
                'Excel Sortie': best_match,
                'Similarity %': f"{best_similarity:.1%}",
                'Image Number': formatted_num,
                'Sortie Landesamt': sortie_landesamt,
                'Match Status': 'MATCHED' if match_found else 'NO MATCH',
                'File Path': os.path.dirname(img_path)
            })

            if match_found:
                new_name = f"{formatted_num}{sortie_landesamt}{file_ext}"
                new_path = os.path.join(os.path.dirname(img_path), new_name)

                print(f"   🖼️ Image {img_num}: ✅ MATCH → {new_name}")

                try:
                    os.rename(img_path, new_path)
                    success_count += 1
                    print("   🎉 Renamed successfully!")
                except Exception as e:
                    print(f"   ❌ Rename failed: {str(e)}")
                    error_count += 1
            else:
                print(f"   🖼️ Image {img_num}: ❌ NO MATCH")
                error_count += 1
    else:
        print(f"   ❌ NO VALID MATCH for folder: {folder_sortie}")
        error_count += len(image_list)

# Results
print(f"\n{'='*60}")
print("📊 COMPREHENSIVE MATCH RESULTS")
print(f"{'='*60}")

if match_results:
    results_df = pd.DataFrame(match_results)
    display(results_df.style.hide(axis='index'))

    matched_count = len([r for r in match_results if r['Match Status'] == 'MATCHED'])
    total_images = len(match_results)

    print(f"\n📈 SUMMARY:")
    print(f"   Matches: {matched_count}/{total_images} ({matched_count/total_images*100:.1f}%)")
    print(f"   Folders processed: {len(images_by_sortie)}")
    print(f"   Images found: {len(all_image_files)}")

print(f"\n{'='*50}")
print("📊 RENAMING SUMMARY")
print(f"{'='*50}")
print(f"✅ Success: {success_count}")
print(f"❌ Failed: {error_count}")
print(f"📁 Total: {total_processed}")

if total_processed > 0:
    print(f"🎯 Success rate: {(success_count/total_processed*100):.1f}%")

📁 Extracted Bestell-ID: 1696
📊 Total records in Excel: 35156
🔍 Found 8 records for ID 1696
📋 Unique sorties in Excel: ['365_BS_2099_12', 'US7_3095', 'US7_3187', 'US7_3188']

🔍 Searching for image files in all subfolders...
📁 Found 8 image files across all subfolders
📂 Found 4 unique parent folders (sorties)

📂 Processing Folder Sortie: 365-BS-2099-12
   📷 Found 2 images
   📊 All possible matches:


,Folder Sortie,Excel Sortie,Similarity
0,365-BS-2099-12,365_BS_2099_12,78.6%
1,365-BS-2099-12,US7_3095,27.3%
2,365-BS-2099-12,US7_3187,18.2%
3,365-BS-2099-12,US7_3188,18.2%


   ✅ Using best match: 365_BS_2099_12 (78.6%)
   📋 Found 2 Excel records
   🖼️ Image 163: ✅ MATCH → 0163#_#365_BS_2099_12#ID1696#09_07_1945.tif
   🎉 Renamed successfully!
   🖼️ Image 164: ✅ MATCH → 0164#_#365_BS_2099_12#ID1696#09_07_1945.tif
   🎉 Renamed successfully!
📂 Processing Folder Sortie: 7-3095
   📷 Found 2 images
   📊 All possible matches:


,Folder Sortie,Excel Sortie,Similarity
0,7-3095,US7_3095,71.4%
1,7-3095,365_BS_2099_12,30.0%
2,7-3095,US7_3187,28.6%
3,7-3095,US7_3188,28.6%


   ✅ Using best match: US7_3095 (71.4%)
   📋 Found 2 Excel records
   🖼️ Image 3160: ✅ MATCH → 3160#00163#US7_3095#ID1696#26_08_1944.tif
   🎉 Renamed successfully!
   🖼️ Image 3161: ✅ MATCH → 3161#00163#US7_3095#ID1696#26_08_1944.tif
   🎉 Renamed successfully!
📂 Processing Folder Sortie: 7-3187
   📷 Found 2 images
   📊 All possible matches:


...

   ✅ Using best match: US7_3187 (71.4%)
   📋 Found 2 Excel records
   🖼️ Image 4053: ✅ MATCH → 4053#00183#US7_3187#ID1696#08_09_1944.tif
   🎉 Renamed successfully!
   🖼️ Image 4054: ✅ MATCH → 4054#00183#US7_3187#ID1696#08_09_1944.tif
   🎉 Renamed successfully!
📂 Processing Folder Sortie: 7-3188
   📷 Found 2 images
   📊 All possible matches:


,Folder Sortie,Excel Sortie,Similarity
0,7-3188,US7_3188,71.4%
1,7-3188,US7_3187,57.1%
2,7-3188,US7_3095,28.6%
3,7-3188,365_BS_2099_12,20.0%


   ✅ Using best match: US7_3188 (71.4%)
   📋 Found 2 Excel records
   🖼️ Image 4074: ✅ MATCH → 4074#00182#US7_3188#ID1696#08_09_1944.tif
   🎉 Renamed successfully!
   🖼️ Image 4075: ✅ MATCH → 4075#00182#US7_3188#ID1696#08_09_1944.tif
   🎉 Renamed successfully!

📊 COMPREHENSIVE MATCH RESULTS


Folder Sortie,Excel Sortie,Similarity %,Image Number,Sortie Landesamt,Match Status,File Path
365-BS-2099-12,365_BS_2099_12,78.6%,0163,#_#365_BS_2099_12#ID1696#09_07_1945,MATCHED,C:\Users\Biplov.Parajuli\Downloads\ImageOrderHeruntergeladen\ID1696 Bammental - LBDB\365-BS-2099-12
365-BS-2099-12,365_BS_2099_12,78.6%,0164,#_#365_BS_2099_12#ID1696#09_07_1945,MATCHED,C:\Users\Biplov.Parajuli\Downloads\ImageOrderHeruntergeladen\ID1696 Bammental - LBDB\365-BS-2099-12
7-3095,US7_3095,71.4%,3160,#00163#US7_3095#ID1696#26_08_1944,MATCHED,C:\Users\Biplov.Parajuli\Downloads\ImageOrderHeruntergeladen\ID1696 Bammental - LBDB\7-3095
7-3095,US7_3095,71.4%,3161,#00163#US7_3095#ID1696#26_08_1944,MATCHED,C:\Users\Biplov.Parajuli\Downloads\ImageOrderHeruntergeladen\ID1696 Bammental - LBDB\7-3095
7-3187,US7_3187,71.4%,4053,#00183#US7_3187#ID1696#08_09_1944,MATCHED,C:\Users\Biplov.Parajuli\Downloads\ImageOrderHeruntergeladen\ID1696 Bammental - LBDB\7-3187
7-3187,US7_3187,71.4%,4054,#00183#US7_3187#ID1696#08_09_1944,MATCHED,C:\Users\Biplov.Parajuli\Downloads\ImageOrderHeruntergeladen\ID1696 Bammental - LBDB\7-3187
7-3188,US7_3188,71.4%,4074,#00182#US7_3188#ID1696#08_09_1944,MATCHED,C:\Users\Biplov.Parajuli\Downloads\ImageOrderHeruntergeladen\ID1696 Bammental - LBDB\7-3188
7-3188,US7_3188,71.4%,4075,#00182#US7_3188#ID1696#08_09_1944,MATCHED,C:\Users\Biplov.Parajuli\Downloads\ImageOrderHeruntergeladen\ID1696 Bammental - LBDB\7-3188



📈 SUMMARY:
   Matches: 8/8 (100.0%)
   Folders processed: 4
   Images found: 8

📊 RENAMING SUMMARY
✅ Success: 8
❌ Failed: 0
📁 Total: 8
🎯 Success rate: 100.0%


## Check for common images, if any, before Moving them in L Drive 

In [14]:
import pandas as pd
from pathlib import Path

def parse_image_info(filename):
    """Extract image number, sortie, and date from filename"""
    try:
        parts = filename.split('.')[0].split('#')
        return parts[0], parts[2], parts[4]  # image_number, sortie, date
    except:
        return None, None, None

def get_sortie_from_folder(folder_name):
    """Extract SORTIE from folder name like '1943_03_04#00042#D_0175'"""
    try:
        return folder_name.split('#')[2]
    except:
        return None

# Define paths
path_a = r"C:\Users\Biplov.Parajuli\Downloads\ImageOrderHeruntergeladen\ID1696 Bammental - LBDB"
path_b = r"W:"

# Scan Path A
print("Scanning Path A...")
images_a = []
for ext in ('.tiff', '.tif', '.ecw'):
    for img_path in Path(path_a).rglob(f'*{ext}'):
        img_num, sortie, date = parse_image_info(img_path.name)
        if img_num and sortie:
            images_a.append({'image_number': img_num, 'sortie': sortie, 'date': date, 'path': img_path})

df_a = pd.DataFrame(images_a)
print(f"Found {len(df_a)} images in Path A")

if df_a.empty:
    print("No images found in Path A")
else:
    # Get unique SORTIES from Path A
    target_sorties = set(df_a['sortie'])
    print(f"Unique SORTIES: {len(target_sorties)}")

    # Scan only relevant folders in Path B
    print("Scanning relevant folders in Path B...")
    images_b = []
    path_b_obj = Path(path_b)

    for folder in path_b_obj.iterdir():
        if folder.is_dir():
            folder_sortie = get_sortie_from_folder(folder.name)
            if folder_sortie in target_sorties:
                for ext in ('.tiff', '.tif', '.ecw'):
                    for img_path in folder.rglob(f'*{ext}'):
                        img_num, sortie, date = parse_image_info(img_path.name)
                        if img_num:
                            images_b.append({'image_number': img_num, 'sortie': sortie, 'date': date, 'path': img_path})

    df_b = pd.DataFrame(images_b)
    print(f"Found {len(df_b)} relevant images in Path B")

    # Find common images
    if not df_b.empty:
        common = pd.merge(df_a, df_b, on=['image_number', 'sortie', 'date'], suffixes=('_A', '_B'))
        print(f"\nCommon images: {len(common)}")
        
        if len(common) > 0:
            for _, row in common.iterrows():
                print(f"{row['image_number']}#_#{row['sortie']}#{row['date']}")
                print(f"  A: {row['path_A']}")
                print(f"  B: {row['path_B']}\n")
    else:
        print("No relevant images found in Path B")

Scanning Path A...
Found 8 images in Path A
Unique SORTIES: 4
Scanning relevant folders in Path B...
Found 696 relevant images in Path B

Common images: 2
0163#_#365_BS_2099_12#09_07_1945
  A: C:\Users\Biplov.Parajuli\Downloads\ImageOrderHeruntergeladen\ID1696 Bammental - LBDB\365-BS-2099-12\0163#_#365_BS_2099_12#ID1696#09_07_1945.tif
  B: W:1945_07_09#2005A#365_BS_2099_12\0163#02005#365_BS_2099_12#ID0001#09_07_1945.tif

0164#_#365_BS_2099_12#09_07_1945
  A: C:\Users\Biplov.Parajuli\Downloads\ImageOrderHeruntergeladen\ID1696 Bammental - LBDB\365-BS-2099-12\0164#_#365_BS_2099_12#ID1696#09_07_1945.tif
  B: W:1945_07_09#2005A#365_BS_2099_12\0164#02005#365_BS_2099_12#ID0001#09_07_1945.tif



## Moving the images and creating a shortcut 

In [15]:
import os
import shutil
from pathlib import Path

def create_shortcut(target_path, shortcut_path):
    """Create a shortcut to the target file"""
    try:
        # For Windows shortcut creation
        import pythoncom
        from win32com.client import Dispatch
        
        shell = Dispatch('WScript.Shell')
        shortcut = shell.CreateShortCut(shortcut_path)
        shortcut.Targetpath = target_path
        shortcut.WorkingDirectory = os.path.dirname(target_path)
        shortcut.save()
        return True
    except:
        # Fallback: copy the file if shortcut creation fails
        try:
            shutil.copy2(target_path, shortcut_path)
            return True
        except:
            return False

def main():
    # Define paths
    path_a = r"C:\Users\Biplov.Parajuli\Downloads\ImageOrderHeruntergeladen\ID1696 Bammental - LBDB"
    path_b = r'W:'
    
    # Step 1: Find all image files in Path A and extract sortie information
    image_files = []
    for root, dirs, files in os.walk(path_a):
        for file in files:
            if file.lower().endswith(('.tif', '.jpg', '.jpeg', '.png', '.tiff')):
                full_path = os.path.join(root, file)
                
                # Extract sortie from filename (part after second #)
                parts = file.split('#')
                if len(parts) >= 3:
                    sortie = parts[2]  # Third part is the sortie
                    image_files.append((full_path, file, sortie))
    
    print(f"Found {len(image_files)} image files in Path A")
    
    # Step 2: Map Path B folders to their sorties
    path_b_folders = {}
    for folder in os.listdir(path_b):
        folder_path = os.path.join(path_b, folder)
        if os.path.isdir(folder_path):
            # Extract sortie from folder name (last part after #)
            parts = folder.split('#')
            if len(parts) >= 2:
                sortie = parts[-1]  # Last part is the sortie
                path_b_folders[sortie] = folder_path
    
    print(f"Found {len(path_b_folders)} folders in Path B")
    
    # Step 3: Move files and create shortcuts
    moved_files = 0
    shortcut_files = 0
    
    for image_path, filename, sortie in image_files:
        if sortie in path_b_folders:
            # Destination path in Path B
            dest_folder = path_b_folders[sortie]
            dest_path = os.path.join(dest_folder, filename)
            
            try:
                # Move file to Path B
                shutil.move(image_path, dest_path)
                moved_files += 1
                print(f"✅ MOVED: {filename} → {dest_folder}")
                
                # Create shortcut back in original location
                shortcut_path = image_path + '.lnk'
                if create_shortcut(dest_path, shortcut_path):
                    shortcut_files += 1
                    print(f"📁 SHORTCUT: Created at {shortcut_path}")
                else:
                    print(f"❌ SHORTCUT FAILED: {filename}")
                    
            except Exception as e:
                print(f"❌ ERROR moving {filename}: {str(e)}")
        else:
            print(f"❌ NO FOLDER FOUND for sortie: {sortie}")
    
    # Summary
    print(f"\n{'='*50}")
    print("📊 OPERATION SUMMARY:")
    print(f"{'='*50}")
    print(f"📁 Total images found: {len(image_files)}")
    print(f"🚚 Files moved to Path B: {moved_files}")
    print(f"📁 Shortcuts created in Path A: {shortcut_files}")
    print(f"📋 Path B folders mapped: {len(path_b_folders)}")

if __name__ == "__main__":
    main()

Found 8 image files in Path A
Found 3180 folders in Path B
✅ MOVED: 0163#_#365_BS_2099_12#ID1696#09_07_1945.tif → W:1945_07_09#2005A#365_BS_2099_12
📁 SHORTCUT: Created at C:\Users\Biplov.Parajuli\Downloads\ImageOrderHeruntergeladen\ID1696 Bammental - LBDB\365-BS-2099-12\0163#_#365_BS_2099_12#ID1696#09_07_1945.tif.lnk
✅ MOVED: 0164#_#365_BS_2099_12#ID1696#09_07_1945.tif → W:1945_07_09#2005A#365_BS_2099_12
📁 SHORTCUT: Created at C:\Users\Biplov.Parajuli\Downloads\ImageOrderHeruntergeladen\ID1696 Bammental - LBDB\365-BS-2099-12\0164#_#365_BS_2099_12#ID1696#09_07_1945.tif.lnk
✅ MOVED: 3160#00163#US7_3095#ID1696#26_08_1944.tif → W:1944_08_26#00163#US7_3095
📁 SHORTCUT: Created at C:\Users\Biplov.Parajuli\Downloads\ImageOrderHeruntergeladen\ID1696 Bammental - LBDB\7-3095\3160#00163#US7_3095#ID1696#26_08_1944.tif.lnk
✅ MOVED: 3161#00163#US7_3095#ID1696#26_08_1944.tif → W:1944_08_26#00163#US7_3095
📁 SHORTCUT: Created at C:\Users\Biplov.Parajuli\Downloads\ImageOrderHeruntergeladen\ID1696 Bamment